In [2]:
!yes | pip uninstall torchvison
!pip install -qU torchvision

yes: standard output: Broken pipe


In [3]:
%%capture
!pip install -U sagemaker

In [11]:
import os
import json
import logging

import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker import get_execution_role

import boto3
from botocore.exceptions import ClientError

## Setup

In [7]:
sess = sagemaker.Session()

print("Default bucket: {}".format(sess.default_bucket()))

role = get_execution_role()

output_path = "s3://" + sess.default_bucket() + "/DEMO-mnist"

Default bucket: sagemaker-us-east-1-875716602731


## PyTorch Estimator

It allows to run a training script using Sagemaker infrastructure on a containerized environemnt. Need to esnure that the framework and python version used is compatible with the latest container. If any errors of non-existing framework or python version appear, ensure to update torchvision and sagemaker

Configuration paramters:
1. entry_point: Training script
2. role: IAM role assumed (needs to be able to read/write on S3, call Sagemaker services etc)
3. instance_type: instance used for the training job. Use "local" if you want to use the notebook's instance for training
4. output_path: S3 bucket URI to save the outputs (model's artifacts and output files)
5. framework_version: The PyTorch version (must be compatible with sagemaker training container)
6. py_version: The Python version (must be compatible with sagemaker training container)

Relevant URLs:
* https://github.com/aws/sagemaker-training-toolkit/blob/master/ENVIRONMENT_VARIABLES.md (Environment Variables)
* https://docs.aws.amazon.com/deep-learning-containers/latest/devguide/dlc-release-notes.html
* https://github.com/aws/sagemaker-python-sdk/tree/master/src/sagemaker/pytorch (sagemaker.pytorch GitHub)
* https://github.com/aws/deep-learning-containers/blob/master/pytorch/training/docker/2.4/py3/cu124/Dockerfile.gpu (sample docker file)

In [8]:
local_mode = False

if local_mode:
    instance_type = "local"
else:
    instance_type = "ml.c5.xlarge"

est = PyTorch(
    entry_point = "train.py",
    source_dir = "code",  # directory of your training script
    role = role,
    framework_version = "2.4.0",
    py_version = "py311",
    instance_type = instance_type,
    instance_count = 1,
    volume_size = 250, # Size in GB of the storage volume to use for storing input and output data during training (default: 30)
    output_path = output_path,
    hyperparameters = {"batch-size": 128, "epochs": 1, "learning-rate": 1e-3, "log-interval": 100}, # passed as command line arguments
)

python train.py --batch-size 100 --epochs 1 --learning-rate 1e-3 --log-interval 100

## Download the data locally

In [12]:
def download_from_s3(data_dir="/root/DeployEndpoint/data", train=True):
    """Download MNIST dataset and convert it to numpy array

    Args:
        data_dir (str): directory to save the data
        train (bool): download training set

    Returns:
        None
    """

    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    if train:
        images_file = "train-images-idx3-ubyte.gz"
        labels_file = "train-labels-idx1-ubyte.gz"
    else:
        images_file = "t10k-images-idx3-ubyte.gz"
        labels_file = "t10k-labels-idx1-ubyte.gz"

    # download objects
    s3 = boto3.client("s3")
    bucket = f"sagemaker-sample-files"
    for obj in [images_file, labels_file]:
        key = os.path.join("datasets/image/MNIST", obj)
        dest = os.path.join(data_dir, obj)
        if not os.path.exists(dest):
            s3.download_file(bucket, key, dest)
    return


download_from_s3("./data", True)
download_from_s3("./data", False)

In [13]:
# Upload to the default bucket

prefix = "DEMO-mnist"
bucket = sess.default_bucket()
loc = sess.upload_data(path="/root/DeployEndpoint/data", bucket=bucket, key_prefix=prefix)

channels = {"training": loc, "testing": loc}

## SM_CHANNEL_{channel_name}

Changing the SM_CHANNEL_{channel_name} container environment parameter. Define the channel_name as key in the dictionary and as value set the S3 URI path

In [15]:
channels = {"training": loc, "testing": loc} # rest of the path is assigned within train.py

## Deploy a training Job

In [17]:
est.fit(inputs=channels)

[11/22/24 21:59:58] INFO     image_uri is not presented, retrieving image_uri based on            ]8;id=904214;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=977597;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py#674\674]8;;\
                             instance_type, framework etc.                                                         

                    INFO     image_uri is not presented, retrieving image_uri based on            ]8;id=972599;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=305376;file:///opt/conda/lib/python3.11/site-packages/sagemaker/image_uris.py#674\674]8;;\
                             instance_type, framework etc.                                                         

                    INFO     Creating training-job with name:                                       ]8;id=452250;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=436558;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#1042\1042]8;;\
                             pytorch-training-2024-11-22-21-59-58-707                                              

2024-11-22 22:00:00 Starting - Starting the training job...
2024-11-22 22:00:15 Starting - Preparing the instances for training...
2024-11-22 22:00:49 Downloading - Downloading input data...
2024-11-22 22:01:19 Downloading - Downloading the training image.....bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2024-11-22 22:02:10,703 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2024-11-22 22:02:10,703 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2024-11-22 22:02:10,704 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2024-11-22 22:02:10,711 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2024-11-22 22:02:10,715 sagemaker_pytorch_container.training INFO     Invoking user training script.
2024-11-22 22:02:12,071 sagemaker-training-toolkit INFO     No GPUs detected (n

## Inspect and store model data

In [18]:
pt_mnist_model_data = est.model_data
print("Model artifact saved at:\n", pt_mnist_model_data)

Model artifact saved at:
 s3://sagemaker-us-east-1-875716602731/DEMO-mnist/pytorch-training-2024-11-22-21-59-58-707/output/model.tar.gz


In [19]:
# We store the variable pt_mnist_model_data in the current notebook kernel.
%store pt_mnist_model_data

Stored 'pt_mnist_model_data' (str)
